# 63. 추가37개 라벨 블라인드 고정 예측
사전에잠근프로토콜로583개기존데이터만학습·보정한다. 추가37개는센서·시각·제품정보만읽고정답은열지않는다. 최종예측을해시와함께저장한뒤64번에서만정답을개봉한다. 재실행으로봉인파일을덮어쓰지못하도록확인한다.
주평가=54온도고정후보+60압력2of3;기준=57온도3of5;보조=기준+압력2of3. 모두47앵커포함. 정답을본뒤모델·임계값·제외기준을변경하지않는다. 과거테스트센서이력을본가능성과저장되지않은조회가능성은감사한계로명시한다.


In [ ]:
from pathlib import Path
import json,hashlib,datetime
import numpy as np,pandas as pd
from sklearn.cluster import KMeans
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
protocol_path=root/'reports/rg3_independent37_locked_protocol.json'
protocol=json.loads(protocol_path.read_text(encoding='utf-8'))
assert protocol['status']=='locked_before_sensor_predictions_and_label_unblinding'
def notebook_source(name):
 return ''.join(json.loads((root/'notebooks'/name).read_text(encoding='utf-8'))['cells'][1]['source'])
# Load old training-only models and data, without opening any extra labels.
s22=notebook_source('22_rg3_nested_signal_ensemble.ipynb')
exec(s22.split('scores=[];thresholds=[];foldlogs=[]')[0].replace('n_jobs=-1','n_jobs=2'))
old_d=d.copy();n_train=len(old_d)
assert n_train==583 and y.sum()==17
metadata=pd.read_csv(tables/'rg3_independent_candidate_keys.csv',parse_dates=['TimeStamp'])
sensors=pd.read_csv(root/'data/moldset_labeled_rg3.csv',nrows=0).columns.drop(['Unnamed: 0','PassOrFail']).tolist()
raw_sensor=pd.read_csv(root/'data/original/labeled_data.csv',usecols=['EQUIP_CD','PART_NAME','TimeStamp']+sensors)
raw_sensor.TimeStamp=pd.to_datetime(raw_sensor.TimeStamp)
raw_sensor=raw_sensor[raw_sensor.PART_NAME.str.contains('RG3',na=False)&raw_sensor.PART_NAME.str.contains('RH',na=False)].copy()
current=metadata.merge(raw_sensor,on=['EQUIP_CD','PART_NAME','TimeStamp'],validate='one_to_one')
assert len(current)==37
current['product']='RG3'
# Check raw-unit agreement on known old shots before applying the frozen model.
old_keys=pd.read_csv(root/'data/audit/rg3_rh_labeled_with_timestamp.csv',usecols=['EQUIP_CD','PART_NAME','TimeStamp'])
old_keys.TimeStamp=pd.to_datetime(old_keys.TimeStamp)
raw_old=old_keys.merge(raw_sensor,on=['EQUIP_CD','PART_NAME','TimeStamp'],validate='one_to_one')
check=old_d[['EQUIP_CD','TimeStamp']+sensors].merge(raw_old,on=['EQUIP_CD','TimeStamp'],validate='one_to_one',suffixes=('_trained','_raw'))
assert len(check)==583
for c in sensors:assert np.allclose(check[c+'_trained'],check[c+'_raw'],equal_nan=True,rtol=1e-5,atol=1e-4),c
# Build existing sensor-only history exactly as notebook10; no new labels loaded.
n10=json.loads((root/'notebooks/10_original_defect_report_revision.ipynb').read_text(encoding='utf-8'))
s10=''.join(next(c for c in n10['cells'] if c['cell_type']=='code')['source'])
exec(s10.split('for feature in features:')[0])
test_index=pd.MultiIndex.from_frame(current[keys])
history_index=pd.MultiIndex.from_frame(observed[keys])
observed=observed.loc[~history_index.isin(test_index)].copy()
observed=pd.concat([observed,current[keys+sensors].assign(sensor_conflict=False)],ignore_index=True)
observed=observed.sort_values(['EQUIP_CD','TimeStamp','product']).reset_index(drop=True)
gaps=observed.groupby('EQUIP_CD').TimeStamp.diff().dt.total_seconds()
switch=observed['product'].ne(observed.groupby('EQUIP_CD')['product'].shift())
observed['history_segment']=(switch|gaps.gt(1800)|gaps.isna()).cumsum()
needed=set(stdcols)|{c for c in old_d.columns if c.endswith('__delta1') or c.endswith('__dev10')}
needed_sensors=sorted({c.split('__')[0] for c in needed})
for c in needed_sensors:
 prior=observed.groupby('history_segment')[c].shift()
 grouped=prior.groupby(observed.history_segment)
 observed[c+'__delta1']=observed[c]-prior
 observed[c+'__past10_std']=grouped.transform(lambda x:x.rolling(10,min_periods=5).std())
 observed[c+'__dev10']=observed[c]-grouped.transform(lambda x:x.rolling(10,min_periods=5).median())
qdata=current[keys+['PART_NAME']].merge(observed,on=keys,validate='one_to_one')
qdata['PassOrFail']=-1
qdata['Reason']='__WITHHELD_NOT_OPENED__'
qdata['comparison_block']='independent_extra37'
d=pd.concat([old_d,qdata],ignore_index=True)
y=np.r_[old_d.PassOrFail.to_numpy(int),np.full(len(qdata),-1,int)]
groups=d.comparison_block
ref=np.arange(n_train);q=np.arange(n_train,len(d))
assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
assert d.TimeStamp.iloc[ref].max()<d.TimeStamp.iloc[q].min()
assert (y[q]==-1).all()
pd.DataFrame({'TimeStamp':qdata.TimeStamp,'missing_sensor_count':qdata[sensors].isna().sum(axis=1),'missing_prior_std_count':qdata[stdcols].isna().sum(axis=1)}).to_csv(tables/'rg3_independent_input_quality.csv',index=False,encoding='utf-8-sig')
# Original 47 innovation expert, now evaluated on truly held-out rows.
s47=notebook_source('47_rg3_innovation_seed_sensitivity.ipynb')
expert_code=s47[s47.index('def expert('):s47.index('records=[]')]
expert_code=expert_code.replace('memo','innovation_cache').replace('np.zeros((len(q),2))','np.zeros((len(q),3))').replace('scores=[[],[]]','scores=[[],[],[]]').replace('[7,101]','[42,7,101]')
innovation_cache={};exec(expert_code)
base_outer=pair_predict(ref,q).max(axis=1)
innovation_outer=expert(ref,q).mean(axis=1)
base_inner=np.full(n_train,np.nan);innovation_inner=np.full(n_train,np.nan)
for b in old_d.comparison_block.unique():
 mask=groups.iloc[ref].eq(b).to_numpy()
 base_inner[mask]=pair_predict(ref[~mask],ref[mask]).max(axis=1)
 innovation_inner[mask]=expert(ref[~mask],ref[mask]).mean(axis=1)
normal=(y[ref]==0)&np.isfinite(innovation_inner)
base_cut=float(np.quantile(base_inner[y[ref]==0],.99,method='higher'))
innovation_cut=float(np.quantile(innovation_inner[normal],.995,method='higher'))
delta=d.Barrel_Temperature_5__delta1.to_numpy(float)
delta_cut=float(np.quantile(delta[ref][normal&np.isfinite(delta[ref])],.95,method='higher'))
base=base_outer>base_cut
anchor47=base|((innovation_outer>innovation_cut)&(delta[q]>delta_cut))
print('Training-only original RF and innovation calibration finished',flush=True)
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neighbors import NearestNeighbors
# Fixed thermal and pressure recipes from54 and58.
mold=(d.Mold_Temperature_3+d.Mold_Temperature_4)/2
barrel=d[[f'Barrel_Temperature_{i}' for i in range(1,7)]].mean(axis=1)
F={
 'surface_thermal_contact':pd.DataFrame({
 'barrel_mold_gap':barrel-mold,
 'mold_side_difference':d.Mold_Temperature_3-d.Mold_Temperature_4,
 'mold_delta_mean':(d.Mold_Temperature_3__delta1+d.Mold_Temperature_4__delta1)/2,
 'mold_dev_mean':(d.Mold_Temperature_3__dev10+d.Mold_Temperature_4__dev10)/2,
 'barrel_mold_delta_gap':d.Barrel_Temperature_5__delta1-(d.Mold_Temperature_3__delta1+d.Mold_Temperature_4__delta1)/2,
 'pressure_per_filling_time':d.Max_Injection_Pressure/d.Filling_Time.where(d.Filling_Time.abs()>1e-6)}),
 'pressure_transition':pd.DataFrame({
 'pressure_minus_switch_delta':d.Max_Injection_Pressure__delta1-d.Max_Switch_Over_Pressure__delta1,
 'speed_delta':d.Max_Injection_Speed__delta1,
 'filling_delta':d.Filling_Time__delta1,
 'pressure_minus_switch_dev10':d.Max_Injection_Pressure__dev10-d.Max_Switch_Over_Pressure__dev10,
 'cushion_delta':d.Cushion_Position__delta1,
 'barrel_mold_delta_gap':d.Barrel_Temperature_5__delta1-(d.Mold_Temperature_3__delta1+d.Mold_Temperature_4__delta1)/2})}
for x in F.values():x.replace([np.inf,-np.inf],np.nan,inplace=True)
s54=notebook_source('54_rg3_pdf_thermal_confirm_gates.ipynb')
function_template=s54[s54.index('def local_residual('):s54.index("models=['two_tree_mean")]
controls={
 'surface_thermal_contact':['Barrel_Temperature_5','Max_Injection_Speed','Filling_Time','Hopper_Temperature'],
 'pressure_transition':['Max_Injection_Speed','Filling_Time','Barrel_Temperature_5','Mold_Temperature_3']}
settings=[('thermal20',20,[42,7,101]),('thermal30',30,[42,7,101]),('thermal40',40,[42,7,101]),('thermal30_alt1',30,[11,23,37]),('thermal30_alt2',30,[13,29,53])]
branches={};calibrations=[];scores=[]
for tag,k,seeds in settings+[('pressure30',30,[42,7,101]),('pressure30_alt1',30,[11,23,37]),('pressure30_alt2',30,[13,29,53])]:
 functions=function_template.replace('k=min(31,len(normal))',f'k=min({k+1},len(normal))').replace('indexes=indexes[indexes!=i][:30]',f'indexes=indexes[indexes!=i][:{k}]').replace('for seed in [42,7,101]:',f'for seed in {seeds}:')
 exec(functions)
 name='surface_thermal_contact' if tag.startswith('thermal') else 'pressure_transition'
 outer=predict(ref,q,name)[:,1]
 inner=np.full(n_train,np.nan);zi=np.full((n_train,6),np.nan)
 for b in old_d.comparison_block.unique():
  mask=groups.iloc[ref].eq(b).to_numpy()
  inner[mask]=predict(ref[~mask],ref[mask],name)[:,1]
  zi[mask]=local_residual(ref[~mask],ref[mask],name)
 ns=(y[ref]==0)&np.isfinite(inner)
 cut=float(np.quantile(inner[ns],.975 if name=='surface_thermal_contact' else .99,method='higher'))
 if name=='surface_thermal_contact':
  zq=local_residual(ref,q,name)
  gate_cut=float(np.quantile(zi[ns,4],.05))
  gate=zq[:,4]<gate_cut
 else:
  dp=np.round(d.Max_Injection_Pressure__delta1.to_numpy(),2)
  ds=np.round(d.Max_Switch_Over_Pressure__delta1.to_numpy(),2)
  dv=np.round(d.Max_Injection_Speed__delta1.to_numpy(),2)
  gate=(dp[q]<0)&(dv[q]>0)&(ds[q]<dp[q]);gate_cut=np.nan
 branches[tag]=(outer>cut)&gate
 calibrations.append(dict(branch=tag,normal_training_count=int(ns.sum()),score_threshold=cut,confirm_threshold=gate_cut,neighbors=k,seeds=str(seeds)))
 for j,i in enumerate(q):scores.append(dict(branch=tag,TimeStamp=d.TimeStamp.iloc[i],score=float(outer[j]),threshold=cut,confirm_pass=int(gate[j]),branch_positive=int(branches[tag][j])))
 print('Finished frozen branch',tag,flush=True)
thermal_vote=np.sum([branches[t[0]] for t in settings],axis=0)
pressure_vote=np.sum([branches[t] for t in ['pressure30','pressure30_alt1','pressure30_alt2']],axis=0)
reference=anchor47|(thermal_vote>=3)
primary=anchor47|branches['thermal30']|(pressure_vote>=2)
secondary=reference|(pressure_vote>=2)
pred=pd.DataFrame({'EQUIP_CD':qdata.EQUIP_CD,'PART_NAME':qdata.PART_NAME,'TimeStamp':qdata.TimeStamp,'reference_prediction':reference.astype(int),'primary_prediction':primary.astype(int),'secondary_prediction':secondary.astype(int),'thermal_votes':thermal_vote,'pressure_votes':pressure_vote,'baseline_prediction':base.astype(int),'anchor47_prediction':anchor47.astype(int)})
assert len(pred)==37 and not pred.duplicated(['EQUIP_CD','PART_NAME','TimeStamp']).any()
path=tables/'rg3_independent37_predictions_SEALED.csv'
assert not path.exists(),'Do not rerun or overwrite sealed predictions after unblinding.'
pred.to_csv(path,index=False,encoding='utf-8-sig')
pd.DataFrame(calibrations).to_csv(tables/'rg3_independent37_training_calibrations.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(scores).to_csv(tables/'rg3_independent37_branch_scores_SEALED.csv',index=False,encoding='utf-8-sig')
seal=pd.DataFrame([dict(sealed_at_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),prediction_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),protocol_sha256=hashlib.sha256(protocol_path.read_bytes()).hexdigest(),new_labels_loaded=False,train_shots=n_train,test_shots=len(pred))])
seal.to_csv(tables/'rg3_independent37_prediction_seal.csv',index=False,encoding='utf-8-sig')
print('Predictions sealed BEFORE extra labels were opened.')
print(pred[['reference_prediction','primary_prediction','secondary_prediction']].sum().to_string())
